# Beginning

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import evaluate
import warnings
import ujson
import json
import os
import sys
import time
import pandas as pd
import torch
from collections import Counter, defaultdict
from tqdm import tqdm
from itertools import chain
# Ignore all warnings
warnings.filterwarnings("ignore")

from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader
from functools import partial
from transformers.integrations import WandbCallback
from datasets import load_dataset
from data_utils import *
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
# Load package to evaluate models
seqeval = evaluate.load("seqeval")

from datasets import Dataset, DatasetDict, load_dataset

/home/cye73/.conda/envs/bioner/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Create dataset for /data/

In [ ]:
tag2label = {
    "ncbi_disease": tag2label_ncbi,
    "bc5cdr": tag2label_bc5cdr,
    "bc5cdr_chemical": tag2label_bc5cdr_chemical,
    "bc5cdr_disease": tag2label_bc5cdr_disease,
    "ebm_pico": tag2label_pico,
    "pico_extraction": tag2label_pico,
    "cometa": tag2label_cometa,
}

dataset_name = 'bc5cdr'
# dataset_name = 'bc5cdr_chemical'
# dataset_name = 'bc5cdr_disease'
# dataset_name = 'ncbi_disease'
# dataset_name = 'medmentions_st21pv'
# dataset_name = 'ebm_pico'
# dataset_name = 'pico_extraction'
# dataset_name = "chemprot"
dataset = load_bigbio_dataset(dataset_name)
docs = dataset_to_documents(dataset)

if dataset_name in ["ebm_pico"] :
    val_split_ids = dataset['train']['document_id'][:187]
    df = dataset_to_df(dataset=dataset, val_split_ids=val_split_ids, dataset_name = dataset_name)
elif dataset_name in ["pico_extraction"] :
    test_split_ids = dataset['train']['document_id'][:40]
    val_split_ids = dataset['train']['document_id'][40:80]
    df = dataset_to_df(dataset=dataset, val_split_ids=val_split_ids, test_split_ids=test_split_ids, dataset_name = dataset_name)
else :
    df = dataset_to_df(dataset=dataset)

In [ ]:
# Convert to Hugging Face Dataset
dataset = DatasetDict()
for split in ['train', 'validation', 'test']:
    if split == 'validation':
        split = 'dev'
    df_split = load_cometa_dataset("/home/cye73/biomedical-entity-linking/bioel/cometa/", version='specific', type='entity', split=split)
    grouped = (
        df_split.groupby(["document_id", "Example"], as_index=False)
        .apply(lambda subdf: pd.Series({"entities": to_entity_list(subdf)}))
        .reset_index(drop=True)
    )
    grouped["id"] = range(len(grouped))
    grouped = grouped[["id", "document_id", "Example", "entities"]]
    dataset[split] = Dataset.from_pandas(grouped)
dataset_df = dataset_to_df(dataset)
docs = dataset_to_documents(dataset, dataset_name='cometa')

In [8]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations'],
        num_rows: 500
    })
})

In [9]:
df[:5]

,document_id,offsets,text,type,db_ids,split,deabbreviated_text,mention_id
0,10027919,"[[0, 16]]",22-oxacalcitriol,[Chemical],[MESH:C051883],validation,None,10027919.1
23,10027919,"[[28, 57]]",secondary hyperparathyroidism,[Disease],[MESH:D006962],validation,None,10027919.2
32,10027919,"[[75, 92]]",low bone turnover,[Disease],[MESH:D001851],validation,None,10027919.3
1,10027919,"[[106, 119]]",renal failure,[Disease],[MESH:D051437],validation,None,10027919.4
5,10027919,"[[133, 143]]",Calcitriol,[Chemical],[MESH:D002117],validation,None,10027919.5


In [185]:
# df['type'].apply(str).unique()

In [6]:
if dataset_name == 'ncbi_disease':
    for idx, row in df.iterrows():
        row['type'][0] = 'Disease'

elif dataset_name == 'bc5cdr_chemical':
    df = df[df['type'].apply(lambda x: x[0] != 'Disease')]
    
elif dataset_name == 'bc5cdr_disease':
    df = df[df['type'].apply(lambda x: x[0] != 'Chemical')]

elif dataset_name == 'ebm_pico':
    for idx, row in df.iterrows():
        if row['type'][0].startswith("Participant"):
            row['type'][0] = 'Participant'
        elif row['type'][0].startswith("Outcome"):
            row['type'][0] = 'Outcome'
        elif row['type'][0].startswith("Intervention"):
            row['type'][0] = 'Intervention'

df[:5]

,document_id,offsets,text,type,db_ids,split,deabbreviated_text,mention_id
0,10027919,"[[0, 16]]",22-oxacalcitriol,[Chemical],[MESH:C051883],validation,None,10027919.1
23,10027919,"[[28, 57]]",secondary hyperparathyroidism,[Disease],[MESH:D006962],validation,None,10027919.2
32,10027919,"[[75, 92]]",low bone turnover,[Disease],[MESH:D001851],validation,None,10027919.3
1,10027919,"[[106, 119]]",renal failure,[Disease],[MESH:D051437],validation,None,10027919.4
5,10027919,"[[133, 143]]",Calcitriol,[Chemical],[MESH:D002117],validation,None,10027919.5


In [7]:
doc_id = list(docs.keys())[0]
from IPython.display import HTML, display
display(HTML(f"<div style='white-space: pre-wrap; word-wrap: break-word;'>{docs[doc_id]}</div>"))

In [8]:
# Apply the replacement
processed_data = process_dataset(docs, df, tag2label[dataset_name])

In [9]:
pmid = df.iloc[0]['document_id']
processed_data[pmid]

{'input': '22-oxacalcitriol suppresses secondary hyperparathyroidism without inducing low bone turnover in dogs with renal failure.\nBACKGROUND: Calcitriol therapy suppresses serum levels of parathyroid hormone (PTH) in patients with renal failure but has several drawbacks, including hypercalcemia and/or marked suppression of bone turnover, which may lead to adynamic bone disease. A new vitamin D analogue, 22-oxacalcitriol (OCT), has been shown to have promising characteristics. This study was undertaken to determine the effects of OCT on serum PTH levels and bone turnover in states of normal or impaired renal function. METHODS: Sixty dogs were either nephrectomized (Nx, N = 38) or sham-operated (Sham, N = 22). The animals received supplemental phosphate to enhance PTH secretion. Fourteen weeks after the start of phosphate supplementation, half of the Nx and Sham dogs received doses of OCT (three times per week); the other half were given vehicle for 60 weeks. Thereafter, the treatment

In [10]:
check_test_length = [pmid for pmid in processed_data.keys() if processed_data[pmid]['split']=='test']
check_valid_length = [pmid for pmid in processed_data.keys() if processed_data[pmid]['split']=='validation']
len(check_test_length), len(check_valid_length)

(500, 500)

In [ ]:
with open(f"../data/{dataset_name}/{dataset_name}.json", "w") as f:
    ujson.dump(processed_data, f, indent=2)

# Create dataset for /data2/

In [13]:
def process_dataset_data2(docs, df, tag2label):
    """
    Function which modifies the abstract to include the entity:
    Input : DCTN4 as a modifier of chronic Pseudomonas aeruginosa infection
    Output : @@DCTN4##T103@@ as a modifier of @@chronic Pseudomonas aeruginosa infection##T038@@
    -------------
    docs : dict {pmid : abstract}
    df : pd.DataFrame
    """
    processed_data = []
    for pmid in docs:
        # Perform replacements based on offsets
        spans = []
        new_df = df[df["document_id"] == pmid]
        for _, row in new_df.iterrows():
            if len(row["offsets"]) > 1:
                continue
            start, end = row["offsets"][0]
            text = row["text"]
            label = tag2label[row["type"][0]]  # label
            tag = row["type"][0]
            span = {
                "start": start,
                "end": end,
                "label": label,
                "tag": tag,
                "text": text,
            }
            spans.append(span)

        processed_data.append({
            "text": docs[pmid],
            "spans": spans,
            "pmid": pmid,
            "split": row["split"],
        })
    return processed_data

In [14]:
processed_data2 = process_dataset_data2(docs, df, tag2label[dataset_name])

In [15]:
processed_data2[0]

{'text': 'Naloxone reverses the antihypertensive effect of clonidine.\nIn unanesthetized, spontaneously hypertensive rats the decrease in blood pressure and heart rate produced by intravenous clonidine, 5 to 20 micrograms/kg, was inhibited or reversed by nalozone, 0.2 to 2 mg/kg. The hypotensive effect of 100 mg/kg alpha-methyldopa was also partially reversed by naloxone. Naloxone alone did not affect either blood pressure or heart rate. In brain membranes from spontaneously hypertensive rats clonidine, 10(-8) to 10(-5) M, did not influence stereoselective binding of [3H]-naloxone (8 nM), and naloxone, 10(-8) to 10(-4) M, did not influence clonidine-suppressible binding of [3H]-dihydroergocryptine (1 nM). These findings indicate that in spontaneously hypertensive rats the effects of central alpha-adrenoceptor stimulation involve activation of opiate receptors. As naloxone and clonidine do not appear to interact with the same receptor site, the observed functional antagonism suggests th

In [ ]:
with open(f"../data2/{dataset_name}/{dataset_name}.json", "w") as f:
    ujson.dump(processed_data2, f, indent=2)

# Experiment

In [4]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer

class DatasetNER(Dataset):
    def __init__(self, data, tokenizer, label2id, split='train', max_length=512):
        """
        Initializes the DatasetNER class.

        Args:
            data (list): List of dictionaries containing 'text' and 'spans'.
            tokenizer (transformers.PreTrainedTokenizer): Tokenizer to use.
            label2id (dict): Mapping from label strings to integer IDs.
            split (str): Dataset split to use ('train', 'validation', 'test').
            max_length (int): Maximum sequence length.
        """
        self.data = [item for item in data if item['split'] == split]
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length
        self.examples = self.process_data()
        
    def process_data(self):
        """
        Processes the data by tokenizing text and aligning labels.

        Returns:
            list: A list of tokenized inputs with aligned labels.
        """
        examples = []
        for item in self.data:
            text = item['text']
            spans = item['spans']
            entities = []
            for span in spans:
                entities.append({
                    'start': span['start'],
                    'end': span['end'],
                    'label': span['tag']
                })
            tokenized_input = self.tokenizer(
                text,
                truncation=True,
                max_length=self.max_length,
                return_offsets_mapping=True,
                padding='max_length'
            )
            labels = self.align_labels(tokenized_input['offset_mapping'], entities)
            tokenized_input["labels"] = labels
            # Remove offset_mapping as it's no longer needed
            tokenized_input.pop("offset_mapping")
            examples.append(tokenized_input)
        return examples

    def align_labels(self, offset_mapping, entities):
        """
        Aligns labels with tokens based on offset mappings.

        Args:
            offset_mapping (list): List of (start, end) tuples for each token.
            entities (list): List of entities with 'start', 'end', and 'label'.

        Returns:
            list: A list of label IDs aligned with the tokens.
        """
        labels = ['O'] * len(offset_mapping)
        for idx, (start, end) in enumerate(offset_mapping):
            if start == end:
                # Special token like [CLS], [SEP], or padding
                labels[idx] = -100
                continue
            for entity in entities:
                entity_start = entity['start']
                entity_end = entity['end']
                entity_label = entity['label']
                # Check if the token is within the entity span
                if start >= entity_start and end <= entity_end:
                    if start == entity_start:
                        labels[idx] = 'B-' + entity_label
                    else:
                        labels[idx] = 'I-' + entity_label
                    break  # Break if the token has been labeled
        # Convert labels to IDs
        labels = [self.label2id[label] if label != -100 else -100 for label in labels]
        return labels

    def __getitem__(self, idx):
        """
        Retrieves an item by index.

        Args:
            idx (int): Index of the item.

        Returns:
            dict: A dictionary containing tokenized inputs and labels.
        """
        return self.examples[idx]

    def __len__(self):
        """
        Returns the total number of examples.

        Returns:
            int: Number of examples.
        """
        return len(self.examples)


In [5]:
from transformers import AutoTokenizer, DataCollatorForTokenClassification

# Sample data (replace this with your actual data)
data_path = '../data2/bc5cdr/bc5cdr.json'
data = ujson.load(open(data_path))

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Define label mappings
label_list = ['O', 'B-Chemical', 'I-Chemical', 'B-Disease', 'I-Disease']  # Add other labels as needed
label2id = {label: idx for idx, label in enumerate(label_list)}

# Create datasets for train, validation, and test
train_dataset = DatasetNER(data, tokenizer, label2id, split='train')
validation_dataset = DatasetNER(data, tokenizer, label2id, split='validation')
test_dataset = DatasetNER(data, tokenizer, label2id, split='test')

# Initialize data collator
data_collator = DataCollatorForTokenClassification(tokenizer)

# Example of creating DataLoaders
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_dataset, batch_size=2, collate_fn=data_collator)
validation_dataloader = DataLoader(validation_dataset, batch_size=2, collate_fn=data_collator)
test_dataloader = DataLoader(test_dataset, batch_size=2, collate_fn=data_collator)

train_dataloader = DataLoader(train_dataset, batch_size=1, collate_fn=data_collator)

# Example batch
for batch in train_dataloader:
    print(batch)
    break


{'input_ids': tensor([[  101,  6583,  4135, 22500,  2063,  7901,  2015,  1996,  3424, 10536,
          4842, 25808,  3512,  3466,  1997, 18856, 10698, 10672,  1012,  1999,
         14477,  5267, 10760, 23355,  1010, 27491, 23760, 25808,  3512, 11432,
          1996,  9885,  1999,  2668,  3778,  1998,  2540,  3446,  2550,  2011,
         26721,  8159,  3560, 18856, 10698, 10672,  1010,  1019,  2000,  2322,
         12702, 13113,  2015,  1013,  4705,  1010,  2001, 26402,  2098,  2030,
         11674,  2011,  6583,  4135, 15975,  1010,  1014,  1012,  1016,  2000,
          1016, 11460,  1013,  4705,  1012,  1996,  1044, 22571, 12184,  3619,
          3512,  3466,  1997,  2531, 11460,  1013,  4705,  6541,  1011, 25003,
          3527,  4502,  2001,  2036,  6822, 11674,  2011,  6583,  4135, 22500,
          2063,  1012,  6583,  4135, 22500,  2063,  2894,  2106,  2025,  7461,
          2593,  2668,  3778,  2030,  2540,  3446,  1012,  1999,  4167, 24972,
          2013, 27491, 23760, 25808,  

# DEBUG

In [6]:
from data_module_perso import create_label2id
tag2label_full = {
    "mm_st21pv": tag2label_st21pv,
    "bc5cdr": tag2label_bc5cdr,
    "ebm_pico": tag2label_pico,
    "pico_extraction": tag2label_pico,
    "ncbi_disease": tag2label_ncbi,
    "mtsamples": tag2label_mtsamples
}
tags = [tag for tag in tag2label_full[dataset_name]]

In [27]:
dataset_name = "mtsamples"
version = "v1"
data_path = f"../data2/{dataset_name}/{dataset_name}.json"
data = ujson.load(open(data_path))

In [28]:
label2id = create_label2id(data, tags)

In [29]:
label2id

{'B-Medical_Problems': 0,
 'B-Tests': 1,
 'B-Treatments': 2,
 'I-Medical_Problems': 3,
 'I-Tests': 4,
 'I-Treatments': 5,
 'O': 6}

In [30]:
tags

['Medical_Problems', 'Treatments', 'Tests']

In [19]:
tag2label_mtsamples

{'Medical_Problems': 0, 'Treatments': 1, 'Tests': 2}